# 🚁 Drone CV — Multi-Platform Detection Pipeline

**Author:** Richard Wirén, Lead Solution Architect — Ericsson 
**Repo:** [github.com/rwiren/drone-cv-detection](https://github.com/rwiren/drone-cv-detection) 
**Patent:** [WO2025034145A1](https://patents.google.com/patent/WO2025034145A1/en)

---

## What this notebook does:
1. **Train** YOLOv8s at `imgsz=1280` on VisDrone + Autel campus data (A100 GPU, ~1-2h)
2. **Validate** on high-resolution drone imagery (no SAHI needed at 1280px)
3. **Process** DJI Avata 360° equirectangular video

### Prerequisites
- Colab Pro with **A100 GPU** runtime
- Upload `autel_labels.zip` (21KB, provided in repo)

In [ ]:
# === 1. SETUP ===
!pip install -q ultralytics
!nvidia-smi | head -4

import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# === 2. DOWNLOAD VISDRONE DATASET ===
import os, zipfile, urllib.request

os.makedirs('datasets/VisDrone/images/train', exist_ok=True)
os.makedirs('datasets/VisDrone/images/val', exist_ok=True)
os.makedirs('datasets/VisDrone/labels/train', exist_ok=True)
os.makedirs('datasets/VisDrone/labels/val', exist_ok=True)

# VisDrone2019-DET (already in YOLO format via ultralytics hub)
from ultralytics import YOLO
# This auto-downloads VisDrone when referenced in the YAML
print('VisDrone will auto-download during training via ultralytics...')
print('If needed manually: https://github.com/VisDrone/VisDrone-Dataset')

In [ ]:
# === 3. UPLOAD AUTEL CAMPUS LABELS ===
from google.colab import files
import zipfile

print('Upload autel_labels.zip (21KB):')
uploaded = files.upload()

with zipfile.ZipFile('autel_labels.zip', 'r') as z:
    z.extractall('autel_data/')
print(f'Extracted: {os.listdir("autel_data/")}')

In [ ]:
# === 4. CREATE DATASET YAML ===
dataset_yaml = """
# Combined VisDrone + Autel Campus
path: /content/datasets/VisDrone
train: images/train
val: images/val

names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

with open('visdrone_autel.yaml', 'w') as f:
    f.write(dataset_yaml)

print('Dataset YAML created')
print('Note: Autel labels will be symlinked into VisDrone train after download')

In [ ]:
# === 5. TRAIN — YOLOv8s at imgsz=1280 ===
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # COCO pretrained base

results = model.train(
    data='VisDrone.yaml',       # Ultralytics auto-downloads VisDrone
    epochs=30,
    imgsz=1280,                 # High-res: handles 4000x3000 natively
    batch=16,                   # A100 80GB handles batch=16 at 1280
    device=0,
    workers=4,
    patience=10,
    project='runs',
    name='visdrone_1280',
    exist_ok=True,
    mosaic=1.0,
    mixup=0.1,
    cos_lr=True,
    plots=True,
)

print(f'\n✅ Training complete!')
print(f'Best model: {results.save_dir}/weights/best.pt')

In [ ]:
# === 6. EVALUATE ===
model = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = model.val(data='VisDrone.yaml', imgsz=1280)

print(f'\n📊 Validation Results:')
print(f'  mAP50 (all):        {metrics.box.map50:.3f}')
print(f'  mAP50-95 (all):     {metrics.box.map:.3f}')
print(f'  mAP50 (car):        {metrics.box.maps[3]:.3f}')
print(f'  mAP50 (pedestrian): {metrics.box.maps[0]:.3f}')

In [ ]:
# === 7. DOWNLOAD TRAINED MODEL ===
from google.colab import files
import shutil

# Copy best weights to easy-to-find location
best_path = f'{results.save_dir}/weights/best.pt'
shutil.copy(best_path, 'visdrone_yolov8s_1280_best.pt')

print(f'Model size: {os.path.getsize("visdrone_yolov8s_1280_best.pt")/1e6:.1f} MB')
files.download('visdrone_yolov8s_1280_best.pt')

---
## 8. Test: Avata 360° Perspective Extraction

Upload an `.LRF` proxy file to test the 360° pipeline on GPU.

In [ ]:
import cv2
import numpy as np

def extract_perspective(equirect, fov_deg=90, yaw_deg=0, pitch_deg=0, out_size=(960, 540)):
    """Extract rectilinear perspective crop from equirectangular frame."""
    h, w = equirect.shape[:2]
    out_w, out_h = out_size
    f = out_w / (2 * np.tan(np.radians(fov_deg) / 2))
    u = np.arange(out_w, dtype=np.float64) - out_w / 2
    v = np.arange(out_h, dtype=np.float64) - out_h / 2
    u, v = np.meshgrid(u, v)
    x, y, z = u, v, np.full_like(u, f)
    norm = np.sqrt(x**2 + y**2 + z**2)
    x, y, z = x/norm, y/norm, z/norm
    cp, sp = np.cos(np.radians(pitch_deg)), np.sin(np.radians(pitch_deg))
    y, z = cp*y - sp*z, sp*y + cp*z
    cy, sy = np.cos(np.radians(yaw_deg)), np.sin(np.radians(yaw_deg))
    x, z = cy*x + sy*z, -sy*x + cy*z
    lon = np.arctan2(x, z)
    lat = np.arcsin(np.clip(y, -1, 1))
    src_x = ((lon / np.pi + 1) / 2 * w).astype(np.float32)
    src_y = ((0.5 - lat / np.pi) * h).astype(np.float32)
    return cv2.remap(equirect, src_x, src_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)

print('extract_perspective() ready')
print('To test: upload .LRF file and run next cell')

In [ ]:
# === Optional: Test 360° detection on uploaded LRF ===
# Uncomment and run after uploading .LRF file

# from google.colab import files
# uploaded = files.upload()  # Upload .LRF file
# lrf_path = list(uploaded.keys())[0]

# cap = cv2.VideoCapture(lrf_path)
# cap.set(cv2.CAP_PROP_POS_FRAMES, int(96 * 30))  # ~1:36 (9m altitude)
# ret, frame = cap.read()
# cap.release()

# model = YOLO('visdrone_yolov8s_1280_best.pt')
# for yaw in range(0, 360, 45):
#     view = extract_perspective(frame, fov_deg=90, yaw_deg=yaw, pitch_deg=-50)
#     results = model(view, conf=0.25, classes=[0], verbose=False)[0]
#     if len(results.boxes) > 0:
#         conf = max(float(b.conf) for b in results.boxes)
#         print(f'  Yaw {yaw:3d}°: {len(results.boxes)} person(s), conf={conf:.2f}')